# 39. Additional SVM and Naive Bayes models
A support vector machine seeks a boundary with a useful margin between classes. A kernel lets it represent nonlinear relationships without explicitly constructing every feature combination. C controls the penalty for margin violations. Different Naive Bayes variants make different assumptions about feature distributions; choose them according to representation, not just naming.

Scale numerical inputs before SVC or SVR unless a justified alternative representation is used. RBF kernels introduce gamma, which controls locality. Large datasets can make kernel methods expensive. GaussianNB models continuous class-conditional features, while MultinomialNB is suited to nonnegative count-like representations. Conditional independence is a simplifying assumption, not a statement that real features are truly unrelated.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
X, y = load_breast_cancer(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
candidate = make_pipeline(StandardScaler(), SVC(C=1, kernel="rbf", gamma="scale")) # Scale before a nonlinear margin model.
scores = cross_val_score(candidate, X_dev, y_dev, cv=3, scoring="accuracy") # Compare through development validation.
print("Mean accuracy:", round(scores.mean(), 3)) # Summarize performance.
assert len(scores) == 3 # Verify evaluation count.

## A small neural-network model-family tour
MLPClassifier and MLPRegressor are feed-forward neural networks trained through backpropagation. hidden_layer_sizes defines layer widths; activation defines nonlinear response; alpha penalizes weights. Numerical scaling matters for optimization. Regression targets with a very large scale may also need a target transformation. Scikit-learn MLPs are useful for learning and bounded tabular experiments; they do not replace a full deep-learning framework or imply GPU training. Avoid assuming that adding neurons improves an already appropriate tree or linear baseline.

In [ ]:
from sklearn.datasets import make_moons, make_regression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error
X, y = make_moons(n_samples=180, noise=0.2, random_state=42)
Xt, Xv, yt, yv = train_test_split(X, y, stratify=y, random_state=42)
classifier = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(12,), solver="lbfgs", max_iter=1000, alpha=0.1, random_state=42)).fit(Xt, yt) # Scale before fitting a bounded network.
print("MLP validation accuracy:", round(classifier.score(Xv, yv), 3))
R, t = make_regression(n_samples=120, n_features=3, noise=5, random_state=42)
Rt, Rv, tt, tv = train_test_split(R, t, random_state=42)
regressor = TransformedTargetRegressor(regressor=make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(8,), solver="lbfgs", max_iter=1000, alpha=0.1, random_state=42)), transformer=StandardScaler()).fit(Rt, tt) # Also scale the regression target, reversing it for prediction.
print("MLP original-unit MAE:", round(mean_absolute_error(tv, regressor.predict(Rv)), 3))
assert len(regressor.predict(Rv)) == len(tv)

### Reading the result
The classifier and regressor solve different tasks on different datasets, so their two scores cannot be compared. lbfgs is a bounded small-data solver choice here. Convergence depends on scaling, iteration budget and settings; a warning should be investigated rather than suppressed or confused with a performance guarantee.

### Transfer exercise
Compare a scaled MLP with a logistic baseline on moons, then with a tree model. Report validation behavior and fit time.

## Your turn
Compare a linear SVC pipeline, an RBF SVC pipeline, and GaussianNB on numerical data. Explain which assumptions differ.

Try the task before opening the solution on the course website.